In [2]:
import json
from pymongo import MongoClient, UpdateOne

# MongoDB 連線設定
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "pc_parts"
COLLECTION_PREFIX = "products"  # Collection 命名方式：products.cpu、products.case...

# 讀取標準化商品資料
with open("standardized_products.json", "r", encoding="utf-8") as f:
    items = json.load(f)

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

imported_count = 0
for item in items:
    category = item["category"]
    collection = db[f"{COLLECTION_PREFIX}.{category}"]

    # 忽略價格為 None 的商品
    if item["price"] is None:
        continue

    # 使用 name + category 作為條件避免重複
    query = {"name": item["name"], "category": category}
    update = {"$set": item}
    collection.update_one(query, update, upsert=True)
    imported_count += 1

print(f"匯入完成，共寫入 {imported_count} 筆商品資料。")

匯入完成，共寫入 2130 筆商品資料。
